In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\Shiwan\OneDrive\Desktop\multi_agent_customer_support_system


In [8]:
# ============================================================
# PROACTIVE AGENT PLAYGROUND
# ============================================================

from uuid import uuid4
from pprint import pprint

from layer_2_proactive_agent.graph.proactive_graph import (
    proactive_graph,
)

from layer_2_proactive_agent.schemas.signal import (
    Signal,
)

from layer_2_proactive_agent.schemas.enums import (
    SignalType,
    SignalSource,
)


In [9]:
# ============================================================
# CHANGE ONLY THIS BLOCK
# ============================================================
TEST_CASE = {
    "customer_id": 1,
    "signal_type": SignalType.INACTIVE_CUSTOMER,
    "signal_source": SignalSource.CRM,
    "signal_context": {}
}

In [10]:
# ============================================================
# BUILD SIGNAL
# ============================================================

signal = Signal(
    signal_id=f"SIG-{uuid4()}",
    customer_id=TEST_CASE["customer_id"],
    signal_type=TEST_CASE["signal_type"],
    signal_source=TEST_CASE["signal_source"],
    signal_context=TEST_CASE["signal_context"],
)

In [11]:
# ============================================================
# INITIAL STATE
# ============================================================

initial_state = {
    "workflow_id": f"WF-{uuid4()}",
    "signal_id": signal.signal_id,
    "signal": signal,

    "customer_profile": None,
    "signal_assessment": None,
    "risk_assessment": None,
    "decision": None,

    "outreach_message": None,
    "escalation_handoff": None,

    "output": None,

    "suppressed": False,
    "suppression_reason": None,

    "current_node": None,
    "workflow_logs": [],
    "errors": [],
}

In [12]:
# ============================================================
# EXECUTE GRAPH
# ============================================================

result = proactive_graph.invoke(
    initial_state,
    config={
        "configurable": {
            "thread_id": str(uuid4())
        }
    }
)

print(type(result))
print(result.keys())

2026-06-05 13:12:28,120 | INFO | proactive_agent | Status=START | Node=VALIDATE_SIGNAL | Workflow=WF-6761cfe2-af41-43a6-b01d-1563e3e315c0
2026-06-05 13:12:28,122 | ERROR | proactive_agent | Status=FAILED | Node=VALIDATE_SIGNAL | Workflow=WF-6761cfe2-af41-43a6-b01d-1563e3e315c0 | Error=Missing or null required signal_context fields for INACTIVE_CUSTOMER: days_inactive


ValueError: State Validation Failed: Missing or null required signal_context fields for INACTIVE_CUSTOMER: days_inactive

In [ ]:
from pprint import pprint

pprint(result["output"])


ProactiveOutput(workflow_id='WF-10f689d4-80ef-45d8-9c98-200c811ac9ba', agent_id='proactive_agent', status=<OutreachStatus.SUPPRESSED: 'SUPPRESSED'>, customer_id=2, signal_assessment=None, risk_assessment=None, decision=None, outreach_message=None, escalation_handoff=None, completed_at=datetime.datetime(2026, 6, 5, 7, 31, 36, 392213, tzinfo=datetime.timezone.utc))


In [ ]:
# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 80)
print("FINAL OUTPUT")
print("=" * 80)

pprint(result["output"])

print("\n" + "=" * 80)
print("WORKFLOW LOGS")
print("=" * 80)

for log in result["workflow_logs"]:
    pprint(log)


FINAL OUTPUT
ProactiveOutput(workflow_id='WF-10f689d4-80ef-45d8-9c98-200c811ac9ba', agent_id='proactive_agent', status=<OutreachStatus.SUPPRESSED: 'SUPPRESSED'>, customer_id=2, signal_assessment=None, risk_assessment=None, decision=None, outreach_message=None, escalation_handoff=None, completed_at=datetime.datetime(2026, 6, 5, 7, 31, 36, 392213, tzinfo=datetime.timezone.utc))

WORKFLOW LOGS
{'message': 'Workflow completed customer=2 status=SUPPRESSED',
 'node': 'response_node',
 'timestamp': '2026-06-05T07:31:36.392213+00:00'}


In [ ]:
# ============================================================
# PROACTIVE AGENT TEST SUITE
# ============================================================

from layer_2_proactive_agent.schemas.enums import (
    SignalType,
    SignalSource,
)

# ============================================================
# TEST 1 — HIGH CHURN (ESCALATION PATH)
# ============================================================
TEST_CASE = {
    "customer_id": 1,
    "signal_type": SignalType.HIGH_CHURN_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": 95,
    },
}

# Expected:
# Signal Severity = HIGH
# Risk Level = CRITICAL
# Action = ESCALATE
# FinalStatus = ESCALATION_REQUIRED


# ============================================================
# TEST 2 — INACTIVE CUSTOMER (OUTREACH PATH)
# ============================================================
TEST_CASE = {
    "customer_id": 2,
    "signal_type": SignalType.INACTIVE_CUSTOMER,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "days_inactive": 120,
    },
}

# Expected:
# Signal Severity = MEDIUM
# Risk Level = MEDIUM
# Action = OUTREACH
# FinalStatus = OUTREACH_CREATED


# ============================================================
# TEST 3 — RECENT NEGATIVE EXPERIENCE
# ============================================================
TEST_CASE = {
    "customer_id": 2,
    "signal_type": SignalType.RECENT_NEGATIVE_EXPERIENCE,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "negative_interactions": 3,
        "ticket_id": "TKT-101",
    },
}

# Expected:
# Severity depends on analyzer rules
# Usually OUTREACH or ESCALATION
# Verify risk calculation


# ============================================================
# TEST 4 — VIP RETENTION RISK
# ============================================================
TEST_CASE = {
    "customer_id": 1,
    "signal_type": SignalType.VIP_RETENTION_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": 99,
        "ltv": 25000,
    },
}

# Expected:
# HIGH/CRITICAL severity
# Enterprise/VIP weighting
# Usually ESCALATION_REQUIRED


# ============================================================
# TEST 5 — INVALID CUSTOMER
# ============================================================
TEST_CASE = {
    "customer_id": 999999,
    "signal_type": SignalType.HIGH_CHURN_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": 95,
    },
}

# Expected:
# customer_context_node failure
# Customer profile not found


# ============================================================
# TEST 6 — SUPPRESSION TEST
# ============================================================
# Run twice back-to-back

TEST_CASE = {
    "customer_id": 1,
    "signal_type": SignalType.HIGH_CHURN_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": 95,
    },
}

# First Run Expected:
# NOT_SUPPRESSED
# Workflow executes normally
#
# Second Run Expected:
# SUPPRESSED
# Route=response_node
# FinalStatus=SUPPRESSED


# ============================================================
# TEST 7 — MISSING CHURN SCORE
# ============================================================
TEST_CASE = {
    "customer_id": 1,
    "signal_type": SignalType.HIGH_CHURN_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {},
}

# Expected:
# validate_signal_node failure
# Missing required signal_context fields:
# churn_score


# ============================================================
# TEST 8 — MISSING DAYS_INACTIVE
# ============================================================
TEST_CASE = {
    "customer_id": 2,
    "signal_type": SignalType.INACTIVE_CUSTOMER,
    "signal_source": SignalSource.CRM,
    "signal_context": {},
}

# Expected:
# validate_signal_node failure
# Missing required signal_context fields:
# days_inactive


# ============================================================
# TEST 9 — MISSING NEGATIVE_INTERACTIONS
# ============================================================
TEST_CASE = {
    "customer_id": 2,
    "signal_type": SignalType.RECENT_NEGATIVE_EXPERIENCE,
    "signal_source": SignalSource.CRM,
    "signal_context": {},
}

# Expected:
# validate_signal_node failure
# Missing required signal_context fields:
# negative_interactions


# ============================================================
# TEST 10 — VIP RETENTION MISSING LTV
# ============================================================
TEST_CASE = {
    "customer_id": 1,
    "signal_type": SignalType.VIP_RETENTION_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": 99,
    },
}

# Expected:
# validate_signal_node failure
# Missing required signal_context fields:
# ltv


# ============================================================
# TEST 11 — INVALID CUSTOMER ID
# ============================================================
TEST_CASE = {
    "customer_id": -1,
    "signal_type": SignalType.HIGH_CHURN_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": 95,
    },
}

# Expected:
# validate_signal_node failure
# Invalid customer_id


# ============================================================
# TEST 12 — NULL BUSINESS FIELD
# ============================================================
TEST_CASE = {
    "customer_id": 1,
    "signal_type": SignalType.HIGH_CHURN_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": None,
    },
}

# Expected:
# validate_signal_node failure
# Missing required signal_context fields:
# churn_score


# ============================================================
# TEST 13 — INVALID SIGNAL TYPE
# ============================================================
TEST_CASE = {
    "customer_id": 1,
    "signal_type": "BAD_SIGNAL",
    "signal_source": SignalSource.CRM,
    "signal_context": {},
}

# Expected:
# validate_signal_node failure
# Unsupported signal_type


# ============================================================
# TEST 14 — ENTERPRISE HIGH RISK CUSTOMER
# ============================================================
TEST_CASE = {
    "customer_id": 1,  # enterprise/premium seeded customer
    "signal_type": SignalType.VIP_RETENTION_RISK,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "churn_score": 100,
        "ltv": 50000,
    },
}

# Expected:
# Risk Level = CRITICAL
# Escalation Candidate = True
# FinalStatus = ESCALATION_REQUIRED


# ============================================================
# TEST 15 — RESPONSE CONTRACT VALIDATION
# ============================================================
TEST_CASE = {
    "customer_id": 2,
    "signal_type": SignalType.INACTIVE_CUSTOMER,
    "signal_source": SignalSource.CRM,
    "signal_context": {
        "days_inactive": 90,
    },
}

# Expected Output Checks:
#
# result["output"].workflow_id
# result["output"].status
# result["output"].customer_id
# result["output"].signal_assessment
# result["output"].risk_assessment
# result["output"].decision
# result["output"].completed_at
#
# No crm_event field should exist.